<h2>Impact (v0.61) - 'Impact (v0.1).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 

'v0.61' makes use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# Define functions for plotting a count of values in a given data field.
int_arr_voys_field_count = lambda field: int_arr_voys[field].value_counts().to_frame().reset_index()
int_dep_voys_field_count = lambda field: int_dep_voys[field].value_counts().to_frame().reset_index()

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.61"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "480"
iso_country = "Mauritius"
year = 2025
gdp = 16.160e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

'Output directory already exists.'

## 1.1 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv("/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_0.1.csv", dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_v0.61.csv".format(iso_country), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Mauritius': 

Number of trade records present in dataset: 14593
Number of partner countries present in dataset: 164
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$6012.2m
Total weight within the dataset: 6121.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,480,Mauritius,392,Japan,000,World,Import,Sea,"Electric accumulators, including separators th...",8507,u,13205.0,39540.015,173374.763,155047.873,173374.763,Electric accumulator,8507 | Electric accumulator,Japan | 8507 | Electric accumulator,85,85 | Electrical machinery,Electrical machinery,Japan | 85 | Electrical machinery,8507,10.3519,0.0966,both,0.228061,16748.002106,[Container],11280.0,7009.064880,117387.833370
1,1,480,Mauritius,344,"China, Hong Kong SAR",000,World,Import,Sea,"Suits, ensembles, jackets, dresses, skirts, di...",6104,u,900.0,249.149,9789.330,9428.179,9789.330,"Suits, ensembles, ja","6104 | Suits, ensembles, ja","China, Hong Kong SAR | 6104 | Suits, ensembles...",61,61 | Apparel and clothing,Apparel and clothing,"China, Hong Kong SAR | 61 | Apparel and clothing",6104,21.6732,0.0461,both,0.025451,451.288113,[Container],8941.0,5555.678111,2507.211491


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 1418
Total size of the fleet: 926 vessels
8 vessel types contained in the dataset: ['Refrigerated bulk' 'Oil tanker' 'Bulk carrier' 'General cargo'
 'Container' 'Chemical tanker' 'Vehicle' 'Liquefied gas tanker']
Total transport work capacity: 402283.8 million deadweight-tonne nautical miles
Total cargo transported: 59353231.9 tonnes
Total transport work: 215142.9 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 21554.7 EJ
Total CO2 produced: 1604693.4 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_arr_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_arr_voys_vess.shape[0] == 0:
                int_arr_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_arr_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
            
            if int_arr_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
                continue
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["lh_levy_usd_"+year] = lh_levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))


print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_im.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.lh_levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['9620 | Monopods, bipods, tr']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 14593
Number of partner countries present in dataset: 164

Total value of trade: US$6012.2m
Total weight within the dataset: 6121.4 million kilograms


Total FEM costs in 2041...

BAU: US$126.8m (2.1% ad-val)
Levy: US$275.1m (4.6% ad-val)
Levy (low-then-high): US$270.5m (4.5% ad-val)
'as-is': US$261.8m (4.4% ad-val)
Brazil: US$268.8m (4.5% ad-val)
LISCR: US$176.9m (2.9% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,480,Mauritius,392,Japan,000,World,Import,Sea,"Electric accumulators, including separators th...",8507,u,13205.0,39540.015,173374.763,155047.873,173374.763,Electric accumulator,8507 | Electric accumulator,Japan | 8507 | Electric accumulator,85,85 | Electrical machinery,Electrical machinery,Japan | 85 | Electrical machinery,8507,10.3519,0.0966,both,0.228061,16748.002106,[Container],11280.0,7009.064880,117387.833370,689.922634,688.146466,688.241793,688.337342,688.337342,689.166998,725.555618,1337.708752,824.505812,788.011468,739.858623,742.648598,708.847387,1305.234815,956.900563,839.417530,769.663302,749.373807,694.177335,1453.122447,1264.860759,1034.785439,918.122949,844.723293,679.786406,1519.226936,1443.729149,1234.475607,1137.929708,912.042327,665.678207,1582.139998,1543.138081,1533.866741,1528.149892,975.920472,651.291370,1544.787052,1488.712534,1484.615973,1607.539874,1249.436199,637.136239,1347.026408,1391.883201,1440.053426,1583.996815,1173.045448,623.335516,1207.169096,1221.313627,1230.776567,1468.095983,1187.610835
1,1,480,Mauritius,344,"China, Hong Kong SAR",000,World,Import,Sea,"Suits, ensembles, jackets, dresses, skirts, di...",6104,u,900.0,249.149,9789.330,9428.179,9789.330,"Suits, ensembles, ja","6104 | Suits, ensembles, ja","China, Hong Kong SAR | 6104 | Suits, ensembles...",61,61 | Apparel and clothing,Apparel and clothing,"China, Hong Kong SAR | 61 | Apparel and clothing",6104,21.6732,0.0461,both,0.025451,451.288113,[Container],8941.0,5555.678111,2507.211491,14.735615,14.697679,14.699715,14.701756,14.701756,14.719476,15.496677,28.571264,17.610091,16.830632,15.802166,15.861755,15.139817,27.877674,20.437826,17.928581,16.438745,16.005395,14.826489,31.036311,27.015350,22.101319,19.609600,18.041903,14.519122,32.448194,30.835686,26.366373,24.304311,19.479727,14.217794,33.791914,32.958897,32.760877,32.638774,20.844060,13.910515,32.994116,31.796457,31.708961,34.334414,26.685907,13.608185,28.770274,29.728341,30.757178,33.831573,25.054326,13.313424,25.783151,26.085255,26.287368,31.356121,25.365419


## 2.4 Inpsect Output

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,136.1,133.5,131.3,129.1,126.8,124.6,122.4,120.1
1,Levy,231.6,226.0,250.3,272.1,275.1,263.9,236.8,216.4
2,Levy (LH),151.0,170.2,232.8,259.0,270.5,261.0,244.8,218.4
3,'as-is',145.1,153.1,183.5,217.4,261.8,252.4,246.2,221.5
4,Brazil,137.2,142.5,166.9,202.7,268.8,272.5,268.8,256.9
5,LISCR,137.8,139.5,155.3,164.5,176.9,212.7,207.2,209.4



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.3,2.2,2.2,2.1,2.1,2.1,2.0,2.0
1,Levy,3.9,3.8,4.2,4.5,4.6,4.4,3.9,3.6
2,Levy (LH),2.5,2.8,3.9,4.3,4.5,4.3,4.1,3.6
3,'as-is',2.4,2.5,3.1,3.6,4.4,4.2,4.1,3.7
4,Brazil,2.3,2.4,2.8,3.4,4.5,4.5,4.5,4.3
5,LISCR,2.3,2.3,2.6,2.7,2.9,3.5,3.4,3.5



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.842,0.826,0.812,0.799,0.785,0.771,0.757,0.743
1,Levy,1.433,1.398,1.549,1.684,1.702,1.633,1.465,1.339
2,Levy (LH),0.935,1.053,1.441,1.603,1.674,1.615,1.515,1.351
3,'as-is',0.898,0.947,1.135,1.345,1.620,1.562,1.523,1.371
4,Brazil,0.849,0.882,1.033,1.254,1.663,1.686,1.663,1.590
5,LISCR,0.853,0.863,0.961,1.018,1.095,1.316,1.282,1.296


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (%)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,95.5,92.4,119.1,143.1,148.2,139.3,114.4,96.3
1,Levy (LH),14.9,36.7,101.5,129.9,143.7,136.4,122.4,98.3
2,'as-is',8.9,19.5,52.2,88.3,134.9,127.8,123.8,101.4
3,Brazil,1.1,8.9,35.6,73.6,141.9,147.9,146.4,136.8
4,LISCR,1.7,5.9,24.0,35.4,50.1,88.2,84.8,89.3



Total change in FEM costs by scenario (%)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,70.1,69.2,90.7,110.8,116.8,111.8,93.4,80.2
1,Levy (LH),10.9,27.5,77.3,100.7,113.3,109.5,100.0,81.8
2,'as-is',6.6,14.6,39.7,68.4,106.4,102.6,101.1,84.4
3,Brazil,0.8,6.7,27.1,57.1,111.9,118.7,119.6,113.9
4,LISCR,1.2,4.4,18.3,27.5,39.5,70.8,69.3,74.3



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.6,1.5,2.0,2.4,2.5,2.3,1.9,1.6
1,Levy (LH),0.2,0.6,1.7,2.2,2.4,2.3,2.0,1.6
2,'as-is',0.1,0.3,0.9,1.5,2.2,2.1,2.1,1.7
3,Brazil,0.0,0.1,0.6,1.2,2.4,2.5,2.4,2.3
4,LISCR,0.0,0.1,0.4,0.6,0.8,1.5,1.4,1.5



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.591,0.572,0.737,0.885,0.917,0.862,0.708,0.596
1,Levy (LH),0.092,0.227,0.628,0.804,0.889,0.844,0.757,0.608
2,'as-is',0.055,0.121,0.323,0.547,0.835,0.791,0.766,0.627
3,Brazil,0.007,0.055,0.220,0.456,0.878,0.915,0.906,0.847
4,LISCR,0.010,0.037,0.149,0.219,0.310,0.545,0.525,0.552


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_v0.61.csv".format(iso_country), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Mauritius': 

Number of trade records present in dataset: 4364
Number of partner countries present in dataset: 109
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1114.7m
Total weight within the dataset: 752.9 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,480,Mauritius,512,Oman,000,World,Export,Sea,"Clothing accessories; made up, knitted or croc...",6117,NaN,0.0,110.154,NaN,5621.050,5621.050,Clothing accessories,6117 | Clothing accessories,Oman | 6117 | Clothing accessories,61,61 | Apparel and clothing,Apparel and clothing,Oman | 61 | Apparel and clothing,6117,18.2553,0.0548,both,0.019597,308.033540,[Container],4651.0,2889.996521,890.215859
1,1,480,Mauritius,372,Ireland,000,World,Export,Sea,"Clothing accessories; made up, knitted or croc...",6117,NaN,0.0,195.115,NaN,16936.146,16936.146,Clothing accessories,6117 | Clothing accessories,Ireland | 6117 | Clothing accessories,61,61 | Apparel and clothing,Apparel and clothing,Ireland | 61 | Apparel and clothing,6117,18.2553,0.0548,both,0.011521,928.100801,[Container],13761.0,8550.686331,7935.898831


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 1423
Total size of the fleet: 934 vessels
8 vessel types contained in the dataset: ['Refrigerated bulk' 'Oil tanker' 'Bulk carrier' 'General cargo'
 'Container' 'Chemical tanker' 'Vehicle' 'Liquefied gas tanker']
Total transport work capacity: 402104.4 million deadweight-tonne nautical miles
Total cargo transported: 59731603.1 tonnes
Total transport work: 215094.9 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 22011.4 EJ
Total CO2 produced: 1647229.2 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_dep_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_dep_voys_vess.shape[0] == 0:
                int_dep_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_dep_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

            if int_dep_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["lh_levy_usd_"+year] = lh_levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.lh_levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(
    trade_ex_na.primaryValue.sum(), 
    round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)
))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['9620 | Monopods, bipods, tr']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 4364
Number of partner countries present in dataset: 109

Total value of trade: US$1114.7m
Total weight within the dataset: 752.9 million kilograms


Total FEM costs in 2041...

BAU: US$17.8m (1.6% ad-val)
Levy: US$40.0m (3.6% ad-val)
Levy (low-then-high): US$39.3m (3.5% ad-val)
'as-is': US$38.6m (3.5% ad-val)
Brazil: US$38.9m (3.5% ad-val)
LISCR: US$25.6m (2.3% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,480,Mauritius,512,Oman,000,World,Export,Sea,"Clothing accessories; made up, knitted or croc...",6117,NaN,0.0,110.154,NaN,5621.050,5621.050,Clothing accessories,6117 | Clothing accessories,Oman | 6117 | Clothing accessories,61,61 | Apparel and clothing,Apparel and clothing,Oman | 61 | Apparel and clothing,6117,18.2553,0.0548,both,0.019597,308.033540,[Container],4651.0,2889.996521,890.215859,5.201768,5.188291,5.189012,5.189735,5.189735,5.196059,5.470502,10.091201,6.216278,5.941214,5.577594,5.598701,5.344451,9.842217,7.217377,6.328389,5.803152,5.650404,5.233781,10.962048,9.543695,7.803835,6.923498,6.370158,5.125209,11.459393,10.891864,9.310440,8.582336,6.872836,5.018767,11.929814,11.638876,11.563331,11.519495,7.361263,4.910222,11.651195,11.226018,11.196088,12.120756,9.415145,4.803581,10.163683,10.496984,10.872692,11.958043,8.857159,4.699459,9.104312,9.211143,9.282106,11.069911,8.953010
1,1,480,Mauritius,372,Ireland,000,World,Export,Sea,"Clothing accessories; made up, knitted or croc...",6117,NaN,0.0,195.115,NaN,16936.146,16936.146,Clothing accessories,6117 | Clothing accessories,Ireland | 6117 | Clothing accessories,61,61 | Apparel and clothing,Apparel and clothing,Ireland | 61 | Apparel and clothing,6117,18.2553,0.0548,both,0.011521,928.100801,[Container],13761.0,8550.686331,7935.898831,46.371565,46.251428,46.257853,46.264294,46.264294,46.320677,48.767222,89.958797,55.415496,52.963420,49.721900,49.910056,47.643523,87.739213,64.339868,56.414914,51.732648,50.370968,46.656949,97.722037,85.078011,69.567894,61.720062,56.787270,45.689077,102.155655,97.096370,82.998648,76.507902,61.268434,44.740188,106.349259,103.755667,103.082217,102.691440,65.622550,43.772560,103.865487,100.075217,99.808405,108.051429,83.932049,42.821898,90.604946,93.576188,96.925464,106.600910,78.957840,41.893691,81.161105,82.113457,82.746063,98.683588,79.812305


## 3.4 Inspect Output

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,19.2,18.8,18.5,18.1,17.8,17.4,17.1,16.8
1,Levy,33.9,33.0,36.6,38.9,40.0,38.9,34.5,31.3
2,Levy (LH),21.6,24.6,33.3,37.5,39.3,38.1,35.7,31.7
3,'as-is',20.7,21.8,26.7,31.5,38.6,37.4,36.5,32.1
4,Brazil,19.4,20.3,24.0,29.3,38.9,40.1,39.9,37.5
5,LISCR,19.5,19.8,22.3,23.8,25.6,31.5,30.2,30.7



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.7,1.7,1.7,1.6,1.6,1.6,1.5,1.5
1,Levy,3.0,3.0,3.3,3.5,3.6,3.5,3.1,2.8
2,Levy (LH),1.9,2.2,3.0,3.4,3.5,3.4,3.2,2.8
3,'as-is',1.9,2.0,2.4,2.8,3.5,3.4,3.3,2.9
4,Brazil,1.7,1.8,2.2,2.6,3.5,3.6,3.6,3.4
5,LISCR,1.8,1.8,2.0,2.1,2.3,2.8,2.7,2.8



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.119,0.116,0.114,0.112,0.110,0.108,0.106,0.104
1,Levy,0.210,0.204,0.227,0.241,0.247,0.241,0.213,0.194
2,Levy (LH),0.134,0.152,0.206,0.232,0.243,0.236,0.221,0.196
3,'as-is',0.128,0.135,0.165,0.195,0.239,0.231,0.226,0.198
4,Brazil,0.120,0.125,0.148,0.181,0.241,0.248,0.247,0.232
5,LISCR,0.121,0.123,0.138,0.147,0.159,0.195,0.187,0.190


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (%)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,14.6,14.1,18.2,20.8,22.2,21.5,17.4,14.5
1,Levy (LH),2.4,5.8,14.8,19.4,21.6,20.6,18.6,14.9
2,'as-is',1.4,3.0,8.2,13.4,20.8,20.0,19.4,15.3
3,Brazil,0.2,1.4,5.5,11.2,21.1,22.7,22.8,20.7
4,LISCR,0.3,1.0,3.8,5.7,7.8,14.1,13.1,13.9



Total change in FEM costs by scenario (%)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,76.1,75.1,98.3,114.6,124.8,123.3,101.6,86.5
1,Levy (LH),12.3,30.6,80.1,106.8,121.2,118.3,108.9,88.7
2,'as-is',7.5,15.9,44.4,73.9,117.0,114.5,113.1,91.2
3,Brazil,1.1,7.6,29.9,61.6,118.8,130.0,133.1,123.3
4,LISCR,1.5,5.3,20.5,31.4,44.0,80.8,76.7,82.9



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.3,1.3,1.6,1.9,2.0,1.9,1.6,1.3
1,Levy (LH),0.2,0.5,1.3,1.7,1.9,1.9,1.7,1.3
2,'as-is',0.1,0.3,0.7,1.2,1.9,1.8,1.7,1.4
3,Brazil,0.0,0.1,0.5,1.0,1.9,2.0,2.0,1.9
4,LISCR,0.0,0.1,0.3,0.5,0.7,1.3,1.2,1.2



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.091,0.087,0.112,0.129,0.137,0.133,0.108,0.090
1,Levy (LH),0.015,0.036,0.092,0.120,0.133,0.128,0.115,0.092
2,'as-is',0.009,0.018,0.051,0.083,0.129,0.124,0.120,0.095
3,Brazil,0.001,0.009,0.034,0.069,0.131,0.140,0.141,0.128
4,LISCR,0.002,0.006,0.023,0.035,0.048,0.087,0.081,0.086
